# P01 — Construcción y evaluación de un modelo de riesgo

**Caso:** Velora Finance

Velora Finance quiere utilizar un modelo de Machine Learning como apoyo en la evaluación de solicitudes de crédito.

El modelo se construye mediante un pipeline preconfigurado para centrar el análisis en su comportamiento, sus errores y la relación entre probabilidad y decisión.

La variable objetivo es:

- `default = 0`: la operación no terminó en impago.
- `default = 1`: la operación terminó en impago.

En P01 no se utiliza `customer_group` como predictor.


## 1. Preparación

Primero cargamos las librerías necesarias.

No es necesario memorizar estos imports. Simplemente proporcionan las funciones que utilizaremos para:

- manipular los datos;
- entrenar el modelo;
- calcular métricas;
- representar los resultados.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    auc,
)

pd.set_option("display.max_columns", None)


### Qué esperamos ver

Si la celda anterior termina mostrando:

`Entorno preparado.`

significa que tenemos disponibles todas las librerías necesarias para continuar.


## 2. Cargar el dataset de Velora Finance

El dataset se genera de forma reproducible a partir de un script almacenado en el repositorio del curso.

Esto garantiza que todos trabajamos con exactamente los mismos datos.


In [ ]:
GENERATOR_URL = "https://raw.githubusercontent.com/pacosepulveda/ia-confiable-explicable-labs/main/resources/generate_velora_credit.py"

namespace = {"__name__": "velora_generator"}

with urllib.request.urlopen(GENERATOR_URL) as response:
    exec(response.read().decode("utf-8"), namespace)

df = namespace["generate_dataset"]()

print("Dataset cargado.")
display(df.head())


### Cómo interpretar el resultado

Cada fila representa una solicitud de crédito.

Algunas columnas importantes son:

- `annual_income`: ingresos anuales;
- `debt_ratio`: ratio de endeudamiento;
- `employment_months`: antigüedad laboral;
- `late_payments_12m`: pagos tardíos durante los últimos 12 meses;
- `requested_amount`: importe solicitado;
- `default`: resultado real de la operación.

La columna `application_id` es solo un identificador.

La columna `customer_group` se conservará para prácticas posteriores, pero **no se utilizará como variable del modelo en P01**.


## 3. ¿Qué tipo de datos tenemos?

Antes de entrenar un modelo conviene saber:

- cuántas solicitudes tenemos;
- qué variables contiene el dataset;
- si existen valores ausentes;
- cómo se distribuye la variable objetivo.


In [ ]:
print("Número de filas y columnas:", df.shape)
print()

df.info()


### Cómo interpretar el resultado

Deberíamos obtener:

- **3.000 solicitudes**;
- 12 columnas;
- todas las variables completas, sin valores nulos.

Esto nos permite centrarnos en la evaluación del modelo sin dedicar tiempo a limpieza de datos.


In [ ]:
df.describe()


### Cómo interpretar el resultado

`describe()` muestra un resumen estadístico de las variables numéricas.

No buscamos memorizar los números. Nos interesa comprobar que:

- las variables están expresadas en escalas diferentes;
- algunas son cantidades monetarias;
- otras son ratios;
- otras son recuentos o meses.

Por ese motivo, más adelante utilizaremos `StandardScaler` antes de entrenar la regresión logística.


## 4. ¿Está equilibrada la variable objetivo?

Vamos a comprobar cuántas operaciones terminaron en impago.


In [ ]:
counts = df["default"].value_counts().sort_index()
percentages = df["default"].value_counts(normalize=True).sort_index() * 100

distribution = pd.DataFrame({
    "Número de casos": counts,
    "Porcentaje": percentages
})

display(distribution)

print(f"Tasa de impago: {df['default'].mean():.2%}")


### Cómo interpretar el resultado

La tasa de impago es aproximadamente del **11,6 %**.

Esto significa que la mayoría de las solicitudes pertenecen a la clase `0`:

> No impago.

Por tanto, las clases están claramente desequilibradas.

Esta observación será importante después: un modelo podría obtener una accuracy elevada simplemente acertando la mayoría de los casos de la clase dominante.

Todavía no sabemos si eso ocurre. Primero entrenaremos el modelo y lo comprobaremos.


## 5. Preparar entrenamiento y test

Separamos los datos en dos partes:

- **80 %** para entrenar el modelo;
- **20 %** para comprobar su comportamiento con datos que no ha utilizado durante el entrenamiento.

Además usamos `stratify=y` para conservar aproximadamente la misma proporción de impagos en ambos conjuntos.


In [ ]:
FEATURES = [
    "age",
    "annual_income",
    "debt_ratio",
    "employment_months",
    "late_payments_12m",
    "credit_history_months",
    "existing_debt",
    "requested_amount",
    "savings_balance",
]

X = df[FEATURES]
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Solicitudes para entrenamiento:", len(X_train))
print("Solicitudes para test:", len(X_test))
print()
print(f"Tasa de impago global: {y.mean():.2%}")
print(f"Tasa de impago train : {y_train.mean():.2%}")
print(f"Tasa de impago test  : {y_test.mean():.2%}")


### Cómo interpretar el resultado

Deberíamos obtener:

- 2.400 solicitudes para entrenamiento;
- 600 solicitudes para test.

Las tasas de impago de train y test deberían ser casi idénticas a la tasa global.

Eso es precisamente lo que buscamos con `stratify=y`.


## 6. Entrenar el modelo base

Utilizamos una **regresión logística**.

No porque sea necesariamente el mejor algoritmo para un sistema real de crédito, sino porque:

- produce probabilidades fáciles de interpretar;
- es rápida;
- es adecuada como modelo base;
- más adelante podremos compararla con modelos menos interpretables.

Antes del modelo aplicamos `StandardScaler` para poner las variables numéricas en escalas comparables.


In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

print("Modelo entrenado correctamente.")


## 7. El modelo produce probabilidades, no decisiones

Para cada solicitud, el modelo calcula una probabilidad estimada de impago.

La decisión binaria aparece después cuando aplicamos un **threshold**.

Comenzaremos con el valor habitual:

`threshold = 0.5`


In [ ]:
proba = model.predict_proba(X_test)[:, 1]

threshold = 0.5
y_pred = (proba >= threshold).astype(int)

preview = pd.DataFrame({
    "probabilidad_impago": proba[:10],
    "prediccion": y_pred[:10],
    "resultado_real": y_test.iloc[:10].values
})

display(preview)


### Cómo interpretar el resultado

Observa la columna `probabilidad_impago`.

El modelo no dice directamente:

- "aprobar";
- "rechazar".

Devuelve valores como:

- 0.08;
- 0.32;
- 0.61;
- 0.84.

Con `threshold = 0.5` convertimos esas probabilidades en:

- probabilidad < 0.5 → `0`;
- probabilidad >= 0.5 → `1`.

Esta diferencia entre **predicción del modelo** y **regla de decisión** será uno de los conceptos centrales de la práctica.


## 8. Primera impresión: Accuracy

Vamos a calcular únicamente la accuracy.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}")


### Cómo interpretar el resultado

Deberíamos obtener aproximadamente:

**89-90 % de accuracy**

A primera vista parece un resultado excelente.

La pregunta es:

> ¿Podemos concluir ya que el modelo funciona bien?

Todavía no.

Tenemos una clase minoritaria de impagos. Necesitamos saber **qué tipos de errores está cometiendo**.


## 9. Abrir la caja: matriz de confusión


In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No impago", "Impago"]
)

disp.plot()
plt.title("Matriz de confusión — threshold = 0.5")
plt.show()

tn, fp, fn, tp = cm.ravel()

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)


### Cómo interpretar el resultado

Con estos datos deberíamos observar aproximadamente:

- muchos verdaderos negativos;
- pocos falsos positivos;
- **muchos falsos negativos**;
- relativamente pocos verdaderos positivos.

En este escenario:

**Falso negativo (FN)**  
El modelo clasifica la operación como de bajo riesgo, pero el cliente termina en impago.

**Falso positivo (FP)**  
El modelo clasifica la operación como de riesgo, pero el cliente realmente no habría incumplido.

Los dos errores tienen consecuencias distintas.

Por eso una única cifra como la accuracy no describe suficientemente el comportamiento del sistema.


## 10. Precision, Recall y F1

Ahora calculamos métricas que permiten analizar mejor la clase `default = 1`.


In [ ]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

basic_metrics = pd.DataFrame({
    "Métrica": ["Accuracy", "Precision", "Recall", "F1"],
    "Valor": [accuracy, precision, recall, f1]
})

display(basic_metrics)


### Cómo interpretar los resultados

**Accuracy**  
Porcentaje total de predicciones correctas.

**Precision**  
De las solicitudes que el modelo clasifica como impago, cuántas realmente terminan en impago.

**Recall**  
De todos los impagos reales, cuántos consigue detectar el modelo.

**F1**  
Combina Precision y Recall en una única métrica.

En esta ejecución, el dato especialmente importante es el **Recall**.

Con `threshold = 0.5`, el modelo detecta únicamente alrededor del **20-25 % de los impagos reales**.

Por tanto:

> Una accuracy cercana al 90 % estaba ocultando que el modelo deja escapar la mayoría de los impagos.

Esta es una de las conclusiones principales de P01.


## 11. Cambiar el threshold

Vamos a comprobar qué ocurre si modificamos el umbral utilizado para convertir probabilidades en decisiones.

La siguiente celda compara automáticamente varios valores de threshold y permite observar cómo cambian:

- Accuracy;
- Precision;
- Recall;
- F1;
- falsos positivos;
- falsos negativos.


In [ ]:
def evaluate_threshold(y_true, probabilities, threshold):
    pred = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()

    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "tn": tn,
    }

thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]

threshold_results = pd.DataFrame(
    [evaluate_threshold(y_test, proba, t) for t in thresholds]
)

display(threshold_results.round(3))


### Cómo interpretar la tabla

Observa especialmente las columnas:

- `recall`;
- `fp`;
- `fn`.

Cuando bajamos el threshold:

- detectamos más impagos;
- aumenta el Recall;
- disminuyen los falsos negativos;
- aumentan los falsos positivos.

Cuando subimos el threshold ocurre lo contrario.

Por ejemplo, aproximadamente:

- con `threshold = 0.5`, detectamos pocos impagos pero tenemos pocos FP;
- con `threshold = 0.2`, detectamos muchos más impagos, pero clasificamos más operaciones solventes como riesgo.

No existe un threshold universalmente correcto.

El threshold depende del coste asociado a cada error y de la política de riesgo de la organización.


## 12. Explorar un threshold concreto

Selecciona un valor con el control `THRESHOLD_ELEGIDO` y ejecuta la celda.

Observa cómo cambia el equilibrio entre:

- Precision y Recall;
- falsos positivos y falsos negativos.

El modelo y sus probabilidades permanecen constantes. Lo que cambia es la regla utilizada para transformar esas probabilidades en una clasificación.


In [ ]:
THRESHOLD_ELEGIDO = 0.30 #@param {type:"slider", min:0.10, max:0.90, step:0.05}

resultado = evaluate_threshold(
    y_test,
    proba,
    THRESHOLD_ELEGIDO
)

metricas_threshold = pd.DataFrame({
    "Métrica": ["Accuracy", "Precision", "Recall", "F1", "FP", "FN"],
    "Valor": [
        resultado["accuracy"],
        resultado["precision"],
        resultado["recall"],
        resultado["f1"],
        resultado["fp"],
        resultado["fn"],
    ]
})

display(metricas_threshold)

pred_elegido = (proba >= THRESHOLD_ELEGIDO).astype(int)
cm_elegida = confusion_matrix(y_test, pred_elegido)

ConfusionMatrixDisplay(
    cm_elegida,
    display_labels=["No impago", "Impago"]
).plot()

plt.title(f"Matriz de confusión — threshold = {THRESHOLD_ELEGIDO:.2f}")
plt.show()


### Interpretación

Compara el resultado con `threshold = 0.5`.

- Un threshold menor suele detectar más impagos y reducir los falsos negativos.
- A cambio, normalmente aumentan los falsos positivos.
- Un threshold mayor suele producir el efecto contrario.

Considera dos escenarios:

**Escenario A**  
El coste de no detectar un futuro impago es elevado.

**Escenario B**  
El coste de clasificar como riesgo una operación solvente es elevado.

El mismo threshold no tiene por qué ser adecuado en ambos casos. La selección del umbral forma parte del diseño y gobierno del sistema, no es una propiedad universal del modelo.


## 13. Profundización — Evaluación a través de múltiples thresholds

ROC-AUC y Precision-Recall permiten analizar el comportamiento del modelo considerando múltiples thresholds, en lugar de evaluar únicamente una regla de decisión concreta.


In [ ]:
roc_auc = roc_auc_score(y_test, proba)
avg_precision = average_precision_score(y_test, proba)

pr_precision, pr_recall, _ = precision_recall_curve(y_test, proba)
pr_auc = auc(pr_recall, pr_precision)

print(f"ROC-AUC          : {roc_auc:.3f}")
print(f"Average Precision: {avg_precision:.3f}")
print(f"PR-AUC           : {pr_auc:.3f}")


### Cómo interpretar estos resultados

**ROC-AUC** mide la capacidad general del modelo para ordenar casos positivos por encima de casos negativos a lo largo de distintos thresholds.

**Average Precision** y la curva **Precision-Recall** son especialmente útiles cuando existe desequilibrio entre clases.

Estas métricas evalúan el modelo de forma más global que una única decisión tomada con `threshold = 0.5`.

Aun así:

> una buena métrica global no determina automáticamente qué threshold debe utilizar una organización.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], "--", label="Clasificador aleatorio")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC")
plt.legend()
plt.show()


In [ ]:
precision_curve, recall_curve, _ = precision_recall_curve(
    y_test,
    proba
)

baseline = y_test.mean()

plt.figure(figsize=(7, 5))
plt.plot(
    recall_curve,
    precision_curve,
    label=f"Average Precision = {avg_precision:.3f}"
)
plt.axhline(
    baseline,
    linestyle="--",
    label=f"Prevalencia = {baseline:.3f}"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall")
plt.legend()
plt.show()


## 14. Revisar errores concretos

Hasta ahora hemos trabajado con métricas agregadas.

Vamos a observar ejemplos concretos de errores cometidos por el modelo con `threshold = 0.5`.


In [ ]:
results = df.loc[X_test.index].copy()

results["prob_default"] = proba
results["prediction"] = y_pred

false_positives = results[
    (results["default"] == 0) &
    (results["prediction"] == 1)
]

false_negatives = results[
    (results["default"] == 1) &
    (results["prediction"] == 0)
]

print("Número de falsos positivos:", len(false_positives))
print("Número de falsos negativos:", len(false_negatives))


In [ ]:
columns_to_show = [
    "application_id",
    "annual_income",
    "debt_ratio",
    "employment_months",
    "late_payments_12m",
    "requested_amount",
    "prob_default",
    "prediction",
    "default"
]

print("Ejemplo de falso positivo:")
display(false_positives[columns_to_show].head(1))

print("Ejemplo de falso negativo:")
display(false_negatives[columns_to_show].head(1))


### Cómo interpretar estos casos

Ahora vemos solicitantes concretos.

Sin embargo, todavía no podemos responder adecuadamente:

> ¿Por qué el modelo ha asignado precisamente esa probabilidad?

Mirar manualmente las variables de entrada no equivale a explicar el modelo.

Esa pregunta será el punto de partida de una práctica posterior dedicada a explicabilidad.


# Conclusiones de P01

Al terminar esta práctica deberíamos haber observado cinco ideas:

**1. Accuracy no es suficiente**

Una accuracy cercana al 90 % puede esconder un Recall muy bajo para la clase que realmente nos preocupa.

**2. Los errores no son equivalentes**

Falsos positivos y falsos negativos producen consecuencias distintas.

**3. El modelo devuelve una probabilidad**

La clasificación binaria aparece cuando aplicamos un threshold.

**4. El threshold forma parte de la decisión**

Cambiarlo modifica el equilibrio entre FP y FN.

**5. No existe una única métrica que determine si un sistema es confiable**

Antes de utilizar un modelo necesitamos analizar su comportamiento en relación con el contexto y el riesgo de cada decisión.

---

En la siguiente práctica podremos plantear una nueva pregunta:

> El modelo puede tener un comportamiento razonable globalmente, pero ¿funciona igual para todos los grupos?
